<a href="https://colab.research.google.com/github/AGMATICABR/meteorologia/blob/main/PEGA_GOES_GOOGLECOLAB_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**PEGA GOES** (Versão Google Colab) Ajuste by José Figueiredo

### Instruções para Execução:

Execute cada Célula Código, aguarde os *deley*.

Para facilitar a execução deste notebook, considere as seguintes orientações:

1.	**Objetivo do Notebook**
Este notebook tem como finalidade baixar, processar e gerar GIFs animados de imagens de satélite GOES para um período de tempo definido.

2.  **Pré-requisitos:**
Certifique-se de que o `ImageMagick` esteja instalado (a primeira célula do notebook se encarrega disso).

3.  **Fluxo de Execução**
Execute as células em sequência, de cima para baixo. A célula inicial instala as dependências, e as células seguintes definem o período de tempo, baixam, processam as imagens JPGs e geram os GIFs.

4.  **Parâmetros Configuráveis**
Na célula de código que contém o script Bash principal, você **deve definir** as variáveis de ano, mês, dia e hora de início (`ai`, `mi`, `di`, `hi`); de fim (`af`, `mf`, `df`, `hf`) e o intervalo em horas entre as imagens (`inc`):

    a)**Datas iniciais**: ( `ai`, `mi`, `di`, `hi`, `af`, `mf`, `df`, `hf`)

    b)**Datas finais**: ( `ai`, `mi`, `di`, `hi`, `af`, `mf`, `df`, `hf`)

    c)**Intervalo**: (`inc`)  ex: `inc=1` para imagens a cada hora, `inc=3` para a cada 3 horas.

5.  **Resultados Gerados:**

    a) **Imagens e GIFs:**
    
      As imagens `.JPGs` e animações `.GIFs` (ex: `anima_vis_03_09.gif`) serão salvos na pasta `/root/jose/images` dentro do ambiente do Colab.

    b) **Arquivo Compactado (`goes_images.zip`)**:
    
     Um arquivo `goes_images.zip` contendo todas as imagens e GIFs gerados será criado e movido para a pasta `/content`. Você pode encontrá-lo e baixá-lo diretamente pelo navegador de arquivos do Colab (ícone de pasta no lado esquerdo da tela).

6.  **Tempo de Execução:**

     A execução do script pode levar alguns minutos, dependendo do período de tempo e da quantidade de dados a serem baixados e processados.




In [ ]:
%%bash
# Instala o ImageMagick para que o comando 'convert' esteja disponível
apt-get update
apt-get install imagemagick -y

echo "ImageMagick instalado com sucesso."

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [4,685 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,172 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,584 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.8 MB]
Get:14 http://security.ubuntu.com/u

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
%%bash
######################################################################
### ESTE SCRIPT PEGA DADOS DO SATELITE GOES DO SITE DO CPTEC/INPE  ###
### Exercício de Aula de Tratamento de Dados                       ###
###                                                                ###
### CTMET/IFSC                                                     ###
###                                                                ###
### Mudança do Satélite GOOS16 para GOES19 no dia 07/04/2025       ###
###                                                                ###
### Elaborado:          MARIO QUADRO       	    - FEB, 26th 2023   ###
### Adaptado:           PROFESSOR QUERIDO       - ABR, 15th 2025   ###
### Ajustado Colab:     ALUNO VETERANO JOSÉ     - AUG, 11th 2026   ###
######################################################################
#
export LC_NUMERIC=en_US.UTF-8
#
###----------------------------------------------------------------###
### SETANDO PATHS                                                  ###
###----------------------------------------------------------------###
#
path_scr=$HOME/jose
path_arq=$path_scr/images
#
mkdir -p $HOME/jose
mkdir -p $path_scr
mkdir -p $path_arq
#
cd $path_arq
#
rm -rf $path_arq/*.jpg*
rm -rf $path_arq/*.gif*
#
###--------------------------------------------------------------------###
### Criar as variáveis meses                                           ###
###--------------------------------------------------------------------###
#
mo=(' ' 'Jan' 'Feb' 'Mar' 'Apr' 'May' 'Jun' 'Jul' 'Aug' 'Sep' 'Oct' 'Nov' 'Dec')
mt=(' ' 'JAN' 'FEB' 'MAR' 'APR' 'MAY' 'JUN' 'JUL' 'AUG' 'SEP' 'OCT' 'NOV' 'DEC')
#
###--------------------------------------------------------------------###
### faz loop nos arquivos e pega data inicial                          ###
###--------------------------------------------------------------------###
#
# Define Data Inicial - FAÇA A ALTERAÇÃO CONFORME DATA
#
ai=2026
mi=08
di=20
hi=00
#
# Define Data Final  -  FAÇA A ALTERAÇÃO CONFORME DATA
#
af=2026
mf=08
df=20
hf=23
#
# Define Intervalo em horas para pegar as imagens
#
inc=1   #(ALTERE por exemplo: 1 para cada hora, 3 para cada 3 horas, etc.)
#
#
##--------------------------------------------------------------------###
### Define Hora Inicial (si), Horário Final (sf) e Faz a conta de     ###
### quantos horários terá o loop (cont_horas)                         ###
###-------------------------------------------------------------------###
#

si=`date -u --date="$ai$mi$di $hi" +%s`;# Linux
#si=`date -j -v +0H -f "%Y%m%d%H" "$ai$mi$di$hi" +%s`;# mac

sf=`date -u --date="$af$mf$df $hf" +%s`
#sf=`date -j -v +0H -f "%Y%m%d%H" "$af$mf$df$hf" +%s`;# mac
#
echo $si
echo $sf
#
#cont_horas=`echo "scale=0; ($sf - $si)/3600/24/30" | bc -l`
cont_horas=`echo "scale=0; 1 + ($sf - $si)/(3600*$inc)" | bc -l`
loop_anim=$((cont_horas-1))
#
echo "Data Inicial         -> "$ai $mi $di $hi
echo "data Final           -> "$af $mf $df $hf
echo "No Horas             -> "$cont_horas
#
#---------------------------------------------------------------------------#
# Loop para juntar os dados e Gerar  ctl e roda gribmap                     #
#---------------------------------------------------------------------------#
#
# faz o loop do período de dados
for (( i = 0; i < $cont_horas; i++ ))
do
  j=$((i*inc))
#
  data=`date -u --date="$ai$mi$di $hi + $j hours" +%Y%m%d%H`'00';# Linux
  datv=`date -u --date="$ai$mi$di $hi + $j hours" +%Y%m%d`;# Linux
  ano=`date -u --date="$ai$mi$di $hi + $j hours" +%Y`;# Linux
  mes=`date -u --date="$ai$mi$di $hi + $j hours" +%m`;# Linux
  dia=`date -u --date="$ai$mi$di $hi + $j hours" +%d`;# Linux
  hor=`date -u --date="$ai$mi$di $hi + $j hours" +%H`;# Linux
  #
  #  data=`date -j -v +${j}H -f "%Y%m%d%H" "$ai$mi$di$hi" +%Y%m%d%H`'00';# mac
  #  ano=`date -j -v +${j}H -f "%Y%m%d%H" "$ai$mi$di$hi" +%Y`;# mac
  #  mes=`date -j -v +${j}H -f "%Y%m%d%H" "$ai$mi$di$hi" +%m`;# mac
  #  dia=`date -j -v +${j}H -f "%Y%m%d%H" "$ai$mi$di$hi" +%d`;# mac
  #  hor=`date -j -v +${j}H -f "%Y%m%d%H" "$ai$mi$di$hi" +%H`;# mac
  #
  echo "Data -> "$data $ano $mes $dia $hor
  #
  # faz o loop dos cacnais das imagens
  #
  for (( k = 1; k <= 3; k++ ))
  do
    #
    # Define o Canal da Imagem
    #
    # Visível
    if  [ $k -eq 1 ]; then
     # Mudança do Satélite GOES16 para GOES19 no dia 07/04/2025
     if  [ $datv -le 20250407 ]; then
      img=S11635368 ; sat=16
     else
      img=S11161203 ; sat=19
     fi
     dir=ams_ret_ch03_baixa ; canal=vis ; ch=03 ; fi
    # Vapor
    if  [ $k -eq 2 ]; then
     # Mudança do Satélite GOES16 para GOES19 no dia 07/04/2025
     if  [ $datv -le 20250407 ]; then
      img=S11635378 ; sat=16
     else
      img=S11161208 ; sat=19
     fi
     dir=ams_ret_ch08_baixa ; canal=vap ; ch=08 ; fi
    # Infravermelho
    if  [ $k -eq 3 ]; then
     # Mudança do Satélite GOES16 para GOES19 no dia 07/04/2025
     if  [ $datv -le 20250407 ]; then
      img=S11635388 ; sat=16
     else
      img=S11161213 ; sat=19
     fi
     dir=ams_ret_ch13_baixa ; canal=ir  ; ch=13 ; fi
    #
    # Tenta baixar a imagem
    wget -q https://satelite.cptec.inpe.br/repositoriogoes/goes${sat}/goes${sat}_web/${dir}/${ano}/${mes}/${img}_${data}.jpg

    # Verifica se o download foi bem-sucedido
    if [ -f "${img}_${data}.jpg" ]; then
      # Reduz o tamanho da Imagem para poder gerar a animação
      convert -resize 50% ${img}_$data.jpg ${img}_$data.jpg
      #
      # Escreve legenda na Imagem
      #
      convert -fill white -draw "rectangle 48,60 1000,120" ${img}_$data.jpg -fill black -pointsize 80 -draw 'text 75,120 "CH'${ch}' '$dia'/'$mes'/'$ano' '$hor' UTC"' ${img}.jpg
      #
      mv ${img}.jpg ${img}_$data.jpg
    else
      echo "Aviso: Imagem ${img}_${data}.jpg não encontrada ou falha no download. Pulando processamento."
    fi

    #
    #---------------------------------------------------------------------------#
    # Gera Animação das Imagens do Satélite GOES-16                             #
    #---------------------------------------------------------------------------#
    #
    if  [ $i -eq $loop_anim ]; then
     # Verifica se há imagens para o canal antes de tentar criar a animação
     if ls ${img}_*.jpg 1> /dev/null 2>&1; then
       echo "======================================================"
       echo "Wait.. Gerando Animacao das Imagens do GOES (canal ${canal})......"
       echo "======================================================"
       convert -delay 70 -loop 0 ${img}_*.jpg anima_${canal}_${ch}_${dia}.gif
     else
       echo "Aviso: Nenhuma imagem encontrada para criar a animação do canal ${canal}."
     fi
    fi
  done

done; # termina loop das estacoes
#
pwd
#---------------------------------------------------------------------------#
# Encerra o script                                                          #
#---------------------------------------------------------------------------#
#
exit

1787184000
1787266800
Data Inicial         -> 2026 08 20 00
data Final           -> 2026 08 20 23
No Horas             -> 24
Data -> 202608200000 2026 08 20 00
Aviso: Imagem S11161203_202608200000.jpg não encontrada ou falha no download. Pulando processamento.
Data -> 202608200100 2026 08 20 01
Aviso: Imagem S11161203_202608200100.jpg não encontrada ou falha no download. Pulando processamento.
Data -> 202608200200 2026 08 20 02
Aviso: Imagem S11161203_202608200200.jpg não encontrada ou falha no download. Pulando processamento.
Data -> 202608200300 2026 08 20 03
Aviso: Imagem S11161203_202608200300.jpg não encontrada ou falha no download. Pulando processamento.
Data -> 202608200400 2026 08 20 04
Aviso: Imagem S11161203_202608200400.jpg não encontrada ou falha no download. Pulando processamento.
Data -> 202608200500 2026 08 20 05
Aviso: Imagem S11161203_202608200500.jpg não encontrada ou falha no download. Pulando processamento.
Data -> 202608200600 2026 08 20 06
Aviso: Imagem S11161203_

In [ ]:
%%bash
# Navega para o diretório onde as imagens estão salvas
cd /root/jose

# Compacta a pasta de imagens em um arquivo zip
zip -r goes_images.zip images/

# Move o arquivo zip para a pasta /content para que seja visível no navegador de arquivos
mv goes_images.zip /content/goes_images.zip

echo "Arquivo goes_images.zip criado e movido para /content/. Você pode baixá-lo pelo navegador de arquivos do Colab."

  adding: images/ (stored 0%)
  adding: images/S11161208_202608202100.jpg (deflated 1%)
  adding: images/S11161213_202608201700.jpg (deflated 1%)
  adding: images/S11161208_202608200200.jpg (deflated 1%)
  adding: images/S11161208_202608201100.jpg (deflated 1%)
  adding: images/S11161203_202608201200.jpg (deflated 2%)
  adding: images/S11161208_202608200500.jpg (deflated 1%)
  adding: images/S11161208_202608200800.jpg (deflated 1%)
  adding: images/S11161208_202608200300.jpg (deflated 1%)
  adding: images/S11161208_202608200600.jpg (deflated 1%)
  adding: images/S11161203_202608201600.jpg (deflated 1%)
  adding: images/S11161213_202608200600.jpg (deflated 1%)
  adding: images/S11161203_202608201500.jpg (deflated 1%)
  adding: images/S11161203_202608202100.jpg (deflated 2%)
  adding: images/S11161208_202608201300.jpg (deflated 1%)
  adding: images/S11161203_202608201900.jpg (deflated 1%)
  adding: images/S11161203_202608202000.jpg (deflated 1%)
  adding: images/S11161213_202608200500.jp